# Experimental Deep Agents Pattern

This notebook mirrors `deepagents_pattern_example.py`. It runs a deterministic stub by default and imports `deepagents` only inside the optional live harness builder.

In [ ]:
import os
from typing import Any, Callable, Dict, List

MODE = os.environ.get("DEEPAGENTS_EXAMPLE_MODE", "stub")
MODEL = os.environ.get("DEEPAGENTS_MODEL", "openai:gpt-5-mini")
TASK = "Explain where Deep Agents fit in a LangGraph notebook."


In [ ]:
def lookup_topic(topic: str) -> str:
    facts = {
        "deep agents": "Deep Agents add planning, subagents, and filesystem-style context tools.",
        "langgraph": "LangGraph provides durable graph execution for agent workflows.",
    }
    return facts.get(topic.lower(), f"No canned fact is available for {topic!r}.")


def summarize_findings(text: str) -> str:
    return "Summary: " + text.strip()[:160]


def build_subagents() -> List[Dict[str, Any]]:
    return [
        {
            "name": "researcher",
            "description": "Looks up concise background facts.",
            "system_prompt": "Return concise, source-aware research notes.",
            "tools": [lookup_topic],
        },
        {
            "name": "summarizer",
            "description": "Condenses findings for the final answer.",
            "system_prompt": "Return a compact summary and next action.",
            "tools": [summarize_findings],
        },
    ]


def run_stub(task: str) -> Dict[str, Any]:
    fact = lookup_topic("deep agents")
    summary = summarize_findings(f"{task}: {fact}")
    return {"mode": "stub", "final_output": summary}


In [ ]:
def build_live_agent(model: str, tools: List[Callable[..., str]]):
    from deepagents import create_deep_agent

    return create_deep_agent(
        model=model,
        tools=tools,
        system_prompt=(
            "You are an experimental Deep Agent. Plan before acting, delegate to "
            "subagents when useful, and keep the final response concise."
        ),
        subagents=build_subagents(),
    )


def run_live(task: str, model: str) -> Dict[str, Any]:
    if not os.environ.get("OPENAI_API_KEY"):
        return {
            "mode": "live-skipped",
            "final_output": "Set OPENAI_API_KEY before running the live Deep Agents example.",
        }
    try:
        agent = build_live_agent(model, [lookup_topic, summarize_findings])
    except ModuleNotFoundError as exc:
        if exc.name == "deepagents":
            return {
                "mode": "live-skipped",
                "final_output": (
                    "Install the optional 'deepagents' package before running "
                    "the live Deep Agents example."
                ),
            }
        raise
    result = agent.invoke({"messages": [{"role": "user", "content": task}]})
    return {"mode": "live", "final_output": str(result)}


In [ ]:
result = run_live(TASK, MODEL) if MODE == "live" else run_stub(TASK)
print(result["final_output"])
